In [ ]:
import pdfplumber
import re
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor # Changed to ThreadPool for Jupyter stability

# --- Functions remain the same ---

def extract_spatial_value(words, anchor_texts, x_tolerance=30, index=0, exclude_text=None):
    if isinstance(anchor_texts, str):
        anchor_texts = [anchor_texts]
    candidates = [w for w in words if any(t.upper() in w['text'].upper() for t in anchor_texts)]
    valid_anchors = []
    for cand in candidates:
        if exclude_text:
            preceded_by_excluded = any(
                exclude_text.upper() in w['text'].upper() 
                and abs(w['top'] - cand['top']) < 5
                and 0 < (cand['x0'] - w['x1']) < 80
                for w in words
            )
            if preceded_by_excluded: continue
        if "OPP." in cand['text'].upper() and "OPPONENT" in cand['text'].upper(): continue
        valid_anchors.append(cand)
    if not valid_anchors: return None
    valid_anchors.sort(key=lambda x: (x['top'], x['x0']))
    anchor = valid_anchors[0]
    column_digits = [w for w in words if abs(w['x0'] - anchor['x0']) < x_tolerance and w['top'] > anchor['bottom'] and re.match(r'^\d+$', w['text'])]
    column_digits.sort(key=lambda x: x['top'])
    return column_digits[index]['text'] if len(column_digits) > index else None

def get_header_metric(words, label_text):
    label_anchor = next((w for w in words if label_text.upper() in w['text'].upper()), None)
    if not label_anchor: return None
    candidates = [w for w in words if re.match(r'^\d+$', w['text']) and ((abs(w['top'] - label_anchor['top']) < 10 and w['x0'] > label_anchor['x1']) or (abs(w['x0'] - label_anchor['x0']) < 30 and w['top'] > label_anchor['bottom']))]
    candidates.sort(key=lambda x: (x['top'], x['x0']))
    return candidates[0]['text'] if candidates else None

def get_spatial_team_name(words, page_height):
    header_words = [w for w in words if w['top'] < page_height * 0.12 and not any(x in w['text'].upper() for x in ["THROUGH", "GAMES", "OFFICIAL", "PAGE"]) and not re.match(r'^\d{1,2}/\d{1,2}/\d{2,4}$', w['text'])]
    if not header_words: return None
    header_words.sort(key=lambda x: (x['top'], x['x0']))
    lines = []
    current_line = [header_words[0]]
    for i in range(1, len(header_words)):
        if abs(header_words[i]['top'] - header_words[i-1]['top']) < 3:
            current_line.append(header_words[i])
        else:
            lines.append(" ".join([w['text'] for w in current_line]))
            current_line = [header_words[i]]
    lines.append(" ".join([w['text'] for w in current_line]))
    clean = re.split(r'\(|:|Through Games|\(OFFICIAL\)', lines[0])[0].strip()
    if re.match(r"^(\w\s)+\w$", clean): clean = clean.replace(" ", "")
    return clean

def process_single_pdf(pdf_path):
    all_teams = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            words = page.extract_words()
            full_text = " ".join([w['text'] for w in words])
            team_name = get_spatial_team_name(words, page.height)
            if not team_name: continue
            record_match = re.search(r"(\d{1,2}\s*-\s*\d{1,2})", full_text)
            metrics = {
                "Team": team_name,
                "Record": record_match.group(1).replace(" ", "") if record_match else None,
                "Avg_RPI_Win": get_header_metric(words, "Average RPI Win"),
                "Avg_RPI_Loss": get_header_metric(words, "Average RPI Loss"),
                "RPI_Rank_D1": extract_spatial_value(words, "TEAM", index=0),
                "RPI_Rank_NonConf": extract_spatial_value(words, "TEAM", index=1),
                "SOS_D1": extract_spatial_value(words, ["SUCCESS", "STRENGTH"], index=0, exclude_text="OPP"),
                "SOS_NonConf": extract_spatial_value(words, ["SUCCESS", "STRENGTH"], index=1, exclude_text="OPP"),
                "Opp_SOS_D1": extract_spatial_value(words, "OPP.", index=0),
                "Opp_SOS_NonConf": extract_spatial_value(words, "OPP.", index=1),
            }
            for key, pattern in [("NET", r"NET:\s*(\d+)"), ("KPI", r"KPI:\s*(\d+)"), ("SOR", r"SOR:\s*(\d+)"), ("POM", r"POM:\s*(\d+)"), ("SAG", r"SAG:\s*(\d+)"), ("BPI", r"BPI:\s*(\d+)")]:
                match = re.search(pattern, full_text)
                metrics[key] = match.group(1) if match else None
            all_teams.append(metrics)
    return os.path.splitext(os.path.basename(pdf_path))[0], all_teams

def clean_and_reconcile(year_key, raw_data, base_path):
    df = pd.DataFrame(raw_data)
    cols_to_check = [c for c in df.columns if c != 'Team']
    df = df.dropna(subset=cols_to_check, how='all').reset_index(drop=True)
    
    # Matching the exact naming convention for the reference file
    ref_path = os.path.join(base_path, "team_sheets", f"{year_key}_Team_Sheets_Final.csv")
    if not os.path.exists(ref_path):
        print(f"Warning: Reference CSV not found for {year_key} at {ref_path}")
        return df
    
    ref_df = pd.read_csv(ref_path)
    if len(df) != len(ref_df):
        print(f"Mismatch in {year_key}: Parsed {len(df)} teams, Reference has {len(ref_df)}.")
    
    # Map reference names to parsed data
    df['Team'] = ref_df.iloc[:, 0].values[:len(df)] 
    
    assert not df['Team'].duplicated().any(), f"Duplicate names in {year_key}"
    return df

# --- Main Execution Block for Jupyter ---

pdf_dir = "/Users/michaelharoon/Projects/tasty/march-madness/data/raw/pdf/men/team_sheets/"
base_proj_path = "/Users/michaelharoon/Projects/tasty/march-madness/data/"
output_dir = "/Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/"

os.makedirs(output_dir, exist_ok=True)
files = [os.path.join(pdf_dir, f) for f in os.listdir(pdf_dir) if f.lower().endswith(".pdf")]

print(f"Processing {len(files)} files...")

# ThreadPoolExecutor is safe for Jupyter/macOS and still parallelizes well here
with ThreadPoolExecutor() as executor:
    results = list(executor.map(process_single_pdf, files))

for year_key, data in results:
    print(f"Cleaning {year_key}...")
    cleaned_df = clean_and_reconcile(year_key, data, base_proj_path)
    save_path = f"/Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/{year_key}_Cleaned.csv"
    cleaned_df.to_csv(save_path, index=False)
    print(f"Saved: {save_path}")

In [ ]:
import pandas as pd
import os
import re

# Define paths
ref_dir = "/Users/michaelharoon/Projects/tasty/march-madness/data/team_sheets/"
dirty_dir = "/Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/"

for year in range(2005, 2020):
    # Determine which suffix exists for this year
    base_name = None
    if os.path.exists(os.path.join(ref_dir, f"{year}_Team_Sheets_Selection.csv")):
        base_name = f"{year}_Team_Sheets_Selection"
    elif os.path.exists(os.path.join(ref_dir, f"{year}_Team_Sheets_Final.csv")):
        base_name = f"{year}_Team_Sheets_Final"
    
    if not base_name:
        print(f"Skipping {year}: No reference file (_Selection or _Final) found.")
        continue

    ref_path = os.path.join(ref_dir, f"{base_name}.csv")
    # Assuming the dirty file followed the base_name from the previous parsing step
    dirty_path = os.path.join(dirty_dir, f"{base_name}_Cleaned.csv")
    
    if not os.path.exists(dirty_path):
        print(f"Skipping {year}: Dirty file {base_name}_Cleaned.csv not found.")
        continue
    
    # Load data
    df_ref = pd.read_csv(ref_path)
    df_dirty = pd.read_csv(dirty_path)
    
    # 1. Extract the Date from the dirty 'Team' column
    sample_text = str(df_dirty.iloc[0]['Team'])
    # Matches "March 16, 2019" or "MARCH 6, 2005"
    date_match = re.search(r"March\s+(\d{1,2}),\s+(\d{4})", sample_text, re.IGNORECASE)
    
    if date_match:
        day = date_match.group(1).zfill(2)
        extracted_year = date_match.group(2)
        new_filename = f"{extracted_year}-03-{day}_Team_Sheets.csv"
    else:
        print(f"!! Date match failed for {year}. Falling back to year-only name.")
        new_filename = f"{year}-03-XX_Team_Sheets.csv"

    # 2. Row count verification
    if len(df_ref) != len(df_dirty):
        print(f"!! Mismatch in {year}: Ref {len(df_ref)} rows, Dirty {len(df_dirty)} rows. Skipping.")
        continue
    
    # 3. Overwrite messy names with clean names
    df_dirty['Team'] = df_ref['Team'].values
    
    # 4. Save to final destination
    save_path = f'/Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/{new_filename}'
    df_dirty.to_csv(save_path, index=False)
    
    print(f"Processed {year}: {base_name} -> {new_filename}")

print("\nCleanup and renaming complete.")

In [ ]:
import pandas as pd
import os
import xlrd
from openpyxl import load_workbook
print(xlrd.__version__)

def get_unique_columns(directory_path):
    if not os.path.exists(directory_path):
        print(f"Error: The directory {directory_path} does not exist.")
        return set()

    unique_columns = set()
    file_count = 0

    # Iterate through the year-based folders
    for i in range(2003, 2027):
        parent_path = os.path.join(directory_path, f"{i}-team-stats")
        
        # Check if the year folder actually exists before listing
        if not os.path.exists(parent_path):
            continue

        for filename in os.listdir(parent_path):
            file_path = os.path.join(parent_path, filename)
            
            if os.path.isfile(file_path):
                try:
                    # Handle CSV files
                    if filename.endswith('.csv'):
                        df = pd.read_csv(file_path, nrows=0)
                        unique_columns.update(df.columns.tolist())
                        file_count += 1
                    

                    elif filename.endswith('.xls'):
                        try:
                            # Try as true XLS
                            df = pd.read_excel(file_path, engine='xlrd', nrows=0)
                        except Exception:
                            try:
                                # Force openpyxl in read-only mode (bypasses style issues)
                                wb = load_workbook(file_path, read_only=True, data_only=True)
                                ws = wb.active

                                # Extract header row manually
                                header = [cell.value for cell in next(ws.iter_rows(max_row=1))]
                                df = pd.DataFrame(columns=header)

                            except Exception as e:
                                print(f"Could not read {filename}: {e}")
                                continue

                        unique_columns.update(df.columns.tolist())
                        file_count += 1

                except Exception as e:
                    print(f"Could not read {filename}: {e}")

    print(f"Processed {file_count} files (CSV and XLS).")
    return sorted(list(unique_columns))

# --- Execution ---
target_folder = '/Users/michaelharoon/Projects/tasty/march-madness/data'
all_cols = get_unique_columns(target_folder)

print(f"{len(all_cols)} Unique Column Names Found:")
print(60 * "-")
print(*all_cols, sep=",")

In [ ]:
import pathlib
import pyarrow.parquet as pq

base_path = '/Users/michaelharoon/Projects/tasty/march-madness/data/market_data_store'
all_columns = set()

# Recursively find all parquet files
for path in pathlib.Path(base_path).rglob('*.parquet'):
    try:
        # Read only the file metadata/schema
        schema = pq.read_schema(path)
        all_columns.update(schema.names)
    except Exception as e:
        print(f"Could not read {path}: {e}")

print("Distinct Columns:", sorted(list(all_columns)))


In [ ]:
import pandas as pd
import os

def calculate_win_pct(record):
    """Parses 'W-L' string and returns W / (W + L)."""
    try:
        if pd.isna(record) or '-' not in str(record):
            return 0
        wins, losses = map(int, str(record).split('-'))
        if (wins + losses) == 0:
            return 0
        return wins / (wins + losses)
    except (ValueError, ZeroDivisionError):
        return 0

directory_path = '/Users/michaelharoon/Projects/tasty/march-madness/data'

# Process only CSV files in the first level of the directory
for filename in os.listdir(directory_path):
    if filename.endswith(".csv") and filename.startswith("20"):
        file_path = os.path.join(directory_path, filename)
        df = pd.read_csv(file_path)

        df.rename(
            columns={
                'RB_KPI': 'KPI',
                'RB_SOR': 'SOR',
                'PM_BPI': "BPI",
                'PM_POM': "POM",
                'PM_SAG': "SAG"
            },
            inplace=True)

        # Save the updated CSV
        df.to_csv(file_path, index=False)


In [ ]:
import pandas as pd
import os

directory_path = '/Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test'

for filename in os.listdir(directory_path):
    if filename.endswith(".csv") and filename.startswith("20"):
        file_path = os.path.join(directory_path, filename)
        
        try:
            df = pd.read_csv(file_path)
            
            if 'Team' in df.columns:
                # 1. Clean the team names for comparison (strip spaces and uppercase)
                # This ensures "Duke" and "duke " are caught as duplicates
                clean_names = df['Team'].astype(str).str.strip().str.upper()
                
                # 2. Find all instances of duplicates
                # keep=False marks every occurrence of a duplicate as True
                is_duplicate = clean_names.duplicated(keep=False)
                
                if is_duplicate.any():
                    # 3. Filter the original dataframe to show the dupes
                    dupes_found = df[is_duplicate].sort_values(by='Team')
                    
                    print(f"\n[!] {filename}: Found {len(dupes_found)} rows with duplicate team names:")
                    print("-" * 50)
                    # Showing 'Team' and any other relevant columns you might want to see
                    print(dupes_found[['Team']]) 
                    print("-" * 50)
                else:
                    print(f"Clean: {filename}")
                    
        except Exception as e:
            print(f"Error reading {filename}: {e}")

In [ ]:
import pandas as pd
ranks = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MMasseyOrdinals.csv')
ranks[ranks['SystemName']=='POM']

In [ ]:
import pandas as pd

# 1. Load Data
ranks = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MMasseyOrdinals.csv')
ts_2026 = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/team_sheets/2026_Team_Sheet_Final.csv')

# Standardize Massey naming
system_col = 'SystemName' if 'SystemName' in ranks.columns else 'SysteName'

# 2. Identify Massey Historical vs 2026
historical_systems = set(ranks[ranks['Season'] < 2026][system_col].unique())
systems_2026_massey = set(ranks[ranks['Season'] == 2026][system_col].unique())

# Systems that existed before but are missing from 2026 Massey data
dropped_in_2026 = historical_systems - systems_2026_massey

# 3. Analyze 2026 Team Sheet coverage
# Identify columns in Team Sheet that have at least one non-null value
ts_available = [col for col in ts_2026.columns if ts_2026[col].notna().any()]
ts_available_set = set(ts_available)

# Mapping dictionary for known aliases (Team Sheet vs Massey abbreviations)
# Standard Massey codes: SAG (Sagarin), POM (Pomeroy), BPI (ESPN), MOR (Moore), etc.
mapping = {
    'POM': 'POM',
    'SAG': 'SAG',
    'BPI': 'BPI',
    'KPI': 'KPI',
    'SOR': 'SOR',
    'NET_Rank': 'NET',
    'RPI_Rank_D1': 'RPI',
    'PM_T-Rank': 'TRK' # BartTorvik
}

# 4. Logical Comparison
can_fill_dropped = [m for m in dropped_in_2026 if m in ts_available_set or m in mapping.values()]
new_to_2026_ts = ts_available_set - historical_systems

# --- REPORTING ---
print(f"--- MASSEY 2026 GAP ANALYSIS ---")
print(f"Systems active in previous years: {len(historical_systems)}")
print(f"Systems active in 2026 Massey:    {len(systems_2026_massey)}")
print(f"Missing from 2026 Massey:        {len(dropped_in_2026)}")

print(f"\n--- TEAM SHEET RECOVERY POTENTIAL ---")
if can_fill_dropped:
    print(f"Team Sheet can RECOVER these missing Massey systems: {can_fill_dropped}")
else:
    print("Team Sheet does not contain any of the dropped Massey systems.")

print(f"\nNew/Custom features in Team Sheet (not in historical Massey):")
print([c for c in new_to_2026_ts if c not in mapping])

# Detailed check for your "Big Three"
for sys in ['SAG', 'POM', 'BPI']:
    in_massey = sys in systems_2026_massey
    in_ts = sys in ts_available_set
    status = "AVAILABLE" if in_ts or in_massey else "MISSING EVERYWHERE"
    print(f"{sys:3} Status: {status} (Massey: {in_massey}, TeamSheet: {in_ts})")

In [ ]:
import pandas as pd

ords = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MMasseyOrdinals.csv')

summary = (
    ords.groupby('SystemName')
    .agg(
        Seasons=('Season', lambda s: sorted(s.unique().tolist())),
        MinDayNum=('RankingDayNum', 'min'),
        MaxDayNum=('RankingDayNum', 'max'),
    )
    .reset_index()
)

for _, row in summary.iterrows():
    if 2026 not in row['Seasons'] or len(row['Seasons']) <=1:
        continue
    print(f"{row['SystemName']}")
    print(f"  Seasons : {row['Seasons']}")
    print(f"  DayNum  : {row['MinDayNum']} – {row['MaxDayNum']}")
    print()

In [225]:
NAMES = 'BPI SOS KPI SOR'
names = NAMES.split(" ")

ords = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MMasseyOrdinals.csv')

for name in names:
    # Filter for the specific system first
    system_df = ords[ords['SystemName'] == name]
    
    print(f"--- System: {name} ---")
    
    # Group by Season and calculate min/max for RankingDayNum
    season_ranges = system_df.groupby('Season')['RankingDayNum'].agg(['min', 'max'])
    
    print(season_ranges)
    print("\n")
# check: RB_KPI,RB_SOR,PM_BPI,PM_POM,PM_SAG,RB_WAB,PM_T-Rank
# get KPI, POM,WAB, SOR (get from espn lowkey) from warren nolan

--- System: BPI ---
        min  max
Season          
2009     57  133
2010     64  133
2011     79  133
2012    121  133
2013    133  133
2018     77  167
2019     62  154
2020     44  163
2026    149  149


--- System: SOS ---
Empty DataFrame
Columns: [min, max]
Index: []


--- System: KPI ---
        min  max
Season          
2016     65  133
2017     44  133
2018     77  167
2019     62  154
2020     64  163
2024    128  133
2025    114  133


--- System: SOR ---
Empty DataFrame
Columns: [min, max]
Index: []




In [ ]:
ords = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MMasseyOrdinals.csv')
print(ords.columns)
ords.sort_values(by='Season', inplace=True)
print(ords[ords['SystemName'] == 'BPI']['Season'].unique())


Index(['Season', 'RankingDayNum', 'SystemName', 'TeamID', 'OrdinalRank'], dtype='str')
[2009 2010 2011 2012 2013 2018 2019 2020 2026]


In [222]:
ords.to_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MMasseyOrdinals.csv')

In [203]:
import pandas as pd

# Load the data
file_path = '/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MMasseyOrdinals.csv'
ords = pd.read_csv(file_path)

# Filter out the SOS and NC_SOS rows
# We keep rows where SystemName is NOT 'SOS' AND NOT 'NC_SOS'
mask = (ords['SystemName'] == 'SOS') | (ords['SystemName'] == 'NC_SOS')
ords_cleaned = ords[~mask].copy()

# Save the file back to the original location
ords_cleaned.to_csv(file_path, index=False)

print(f"Success: Removed SOS/NC_SOS rows. New row count: {len(ords_cleaned)}")

Success: Removed SOS/NC_SOS rows. New row count: 6676955


In [157]:
import pandas as pd
seasons = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MSeasons.csv')
starts = seasons[['Season','DayZero']]
starts['DayZero']

0     10/29/1984
1     10/28/1985
2     10/27/1986
3     11/02/1987
4     10/31/1988
5     10/30/1989
6     10/29/1990
7     11/04/1991
8     11/02/1992
9     11/01/1993
10    10/31/1994
11    10/30/1995
12    10/28/1996
13    10/27/1997
14    10/26/1998
15    11/01/1999
16    10/30/2000
17    10/29/2001
18    11/04/2002
19    11/03/2003
20    11/01/2004
21    10/31/2005
22    10/30/2006
23    11/05/2007
24    11/03/2008
25    11/02/2009
26    11/01/2010
27    10/31/2011
28    11/05/2012
29    11/04/2013
30    11/03/2014
31    11/02/2015
32    10/31/2016
33    10/30/2017
34    11/05/2018
35    11/04/2019
36    11/02/2020
37    11/01/2021
38    10/31/2022
39    11/06/2023
40    11/04/2024
41    11/03/2025
Name: DayZero, dtype: str

In [158]:
import pandas as pd

# Load the seasons data
seasons = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MSeasons.csv')

def calculate_days_to_april_15(df):
    # Ensure Season is filtered for 2002 onwards
    df = df[df['Season'] >= 2002].copy()
    
    # Convert DayZero to datetime
    # Kaggle MSeasons typically uses MM/DD/YYYY format
    df['DayZero'] = pd.to_datetime(df['DayZero'])
    
    # Construct April 15th for each season
    # We use the 'Season' column as the year
    df['April15'] = pd.to_datetime(df['Season'].astype(str) + '-04-15')
    
    # Calculate the difference in days
    # .dt.days returns the integer count
    df['DaysToApril15'] = (df['April15'] - df['DayZero']).dt.days
    
    return df[['Season', 'DayZero', 'April15', 'DaysToApril15']]

# Execute calculation
result = calculate_days_to_april_15(seasons)

# Display results
print(result.to_string(index=False))

# Optional: Create a dictionary for quick lookup in your other scripts
# day_offset_map = dict(zip(result.Season, result.DaysToApril15))

 Season    DayZero    April15  DaysToApril15
   2002 2001-10-29 2002-04-15            168
   2003 2002-11-04 2003-04-15            162
   2004 2003-11-03 2004-04-15            164
   2005 2004-11-01 2005-04-15            165
   2006 2005-10-31 2006-04-15            166
   2007 2006-10-30 2007-04-15            167
   2008 2007-11-05 2008-04-15            162
   2009 2008-11-03 2009-04-15            163
   2010 2009-11-02 2010-04-15            164
   2011 2010-11-01 2011-04-15            165
   2012 2011-10-31 2012-04-15            167
   2013 2012-11-05 2013-04-15            161
   2014 2013-11-04 2014-04-15            162
   2015 2014-11-03 2015-04-15            163
   2016 2015-11-02 2016-04-15            165
   2017 2016-10-31 2017-04-15            166
   2018 2017-10-30 2018-04-15            167
   2019 2018-11-05 2019-04-15            161
   2020 2019-11-04 2020-04-15            163
   2021 2020-11-02 2021-04-15            164
   2022 2021-11-01 2022-04-15            165
   2023 20

In [160]:
pd.set_option('display.max_rows', None)

bpi = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/raw/archived/bpi/espn_bpi_team_rankings_2013_to_present.csv')
bpi = bpi[bpi['season']==2026]

ts = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/team_sheets/2026-04-01_Team_Sheet_Partial.csv')

df = pd.merge(bpi, ts, left_on='bpi_rank', right_on='PM_BPI')[['team','Team','PM_BPI','bpi_rank']]
df.rename(columns={'Team':'wn_team', 'team':'espn_team'}, inplace=True)

df.iloc[106]

espn_team    Pacific Tigers
wn_team        Saint Thomas
PM_BPI                  131
bpi_rank                131
Name: 106, dtype: object

In [161]:
unsimilar = df[df.apply(lambda x: str(x['wn_team']) not in str(x['espn_team']), axis=1)]
unsimilar.head(5)
# compare it to 2026 team sheet then Map team name ot id then move it to mmassey ordinals

,espn_team,wn_team,PM_BPI,bpi_rank
5,Illinois Fighting Illini,Iowa State,6,6
6,Iowa State Cyclones,Illinois,7,7
7,Purdue Boilermakers,Gonzaga,8,8
8,Gonzaga Bulldogs,Purdue,9,9
9,UConn Huskies,Connecticut,10,10


In [164]:
import os
import re
import shutil

def reorganize_by_season(root_dir, dry_run=True):
    # Matches the date prefix YYYY-MM-DD at the start of the filename
    date_pattern = re.compile(r'^(\d{4})-(\d{2})-(\d{2})')
    
    if not os.path.exists(root_dir):
        print(f"Directory not found: {root_dir}")
        return

    moved_count = 0

    # Walk through the archive
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            match = date_pattern.match(filename)
            if match:
                file_year = int(match.group(1))
                file_month = int(match.group(2))
                
                # Logic: If month is > 4 (May through Dec), it belongs to the NEXT season
                # Otherwise (Jan through April), it stays in the current file_year folder
                season_year = file_year + 1 if file_month > 4 else file_year
                
                # Define the target directory path
                target_subdir = os.path.join(root_dir, str(season_year))
                current_path = os.path.join(dirpath, filename)
                target_path = os.path.join(target_subdir, filename)

                # Skip if it's already in the correct folder
                if os.path.normpath(dirpath) == os.path.normpath(target_subdir):
                    continue

                moved_count += 1
                if not dry_run:
                    # Create the year folder if it doesn't exist
                    os.makedirs(target_subdir, exist_ok=True)
                    shutil.move(current_path, target_path)
                    print(f"MOVED: {filename} -> {season_year}/")
                else:
                    print(f"[DRY RUN] Would move: {current_path} -> {target_path}")

    print(f"\nTotal files identified for relocation: {moved_count}")

# Configuration
archive_path = "/Users/michaelharoon/Projects/tasty/march-madness/data/raw/nitty_gritty/csvs"

# Run the script
reorganize_by_season(archive_path, dry_run=True)


Total files identified for relocation: 0


In [177]:
import pandas as pd
import glob
import os

# Base file and directory
file_initial = '/Users/michaelharoon/Projects/tasty/march-madness/data/raw/nitty_gritty/csvs/2025/thru_games_initial.csv'
directory_2025 = '/Users/michaelharoon/Projects/tasty/march-madness/data/raw/nitty_gritty/csvs/2025/'

def find_most_similar_csv(target_path, search_dir):
    # Load target
    df_target = pd.read_csv(target_path)
    target_cells = df_target.size
    
    results = []
    
    # Get all CSVs in the folder
    all_files = glob.glob(os.path.join(search_dir, "*.csv"))
    
    for file in all_files:
        # Skip the initial file itself
        if os.path.abspath(file) == os.path.abspath(target_path):
            continue
            
        try:
            df_current = pd.read_csv(file)
            
            # Ensure shape matches for a direct cell-to-cell comparison
            if df_target.shape != df_current.shape:
                # Calculate similarity based on shared rows/cols if shapes differ
                # For your case, we assume they match as per previous context
                score = 0.0 
                status = f"Shape Mismatch: {df_current.shape}"
            else:
                # Count matching cells
                # (df1 == df2) returns a boolean mask; .sum().sum() counts True values
                matches = (df_target == df_current).sum().sum()
                score = (matches / target_cells) * 100
                status = f"{score:.2f}% Match"

            results.append({
                'file': os.path.basename(file),
                'score': score,
                'status': status
            })
            
        except Exception as e:
            print(f"Could not process {file}: {e}")

    # Sort by score descending
    sorted_results = sorted(results, key=lambda x: x['score'], reverse=True)
    
    print(f"{'File Name':<40} | {'Match Score'}")
    print("-" * 60)
    for res in sorted_results:
        print(f"{res['file']:<40} | {res['status']}")

if __name__ == "__main__":
    find_most_similar_csv(file_initial, directory_2025)

File Name                                | Match Score
------------------------------------------------------------
thru_games_03_16_2025 (Selections).csv   | 100.00% Match
thru_games_03_15_2025.csv                | 83.33% Match
thru_games_03_14_2025.csv                | 53.22% Match
thru_games_03_13_2025.csv                | 37.14% Match
thru_games_03_12_2025.csv                | 29.03% Match
thru_games_03_10_2025.csv                | 22.40% Match
thru_games_03_11_2025.csv                | 22.32% Match
thru_games_03_09_2025.csv                | 21.68% Match
thru_games_04_07_2025 (Final).csv        | 18.01% Match
thru_games_03_08_2025.csv                | 16.35% Match
thru_games_03_07_2025.csv                | 15.12% Match
thru_games_03_06_2025.csv                | 13.51% Match
thru_games_03_05_2025.csv                | 12.69% Match
thru_games_03_04_2025.csv                | 11.94% Match
thru_games_12_27_2024.csv                | 11.92% Match
thru_games_02_28_2025.csv                | 

In [215]:
# CHECK: season 2015 rankdaynum 62  SOS_D1 ID 1154   rank 324
teams = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MTeams.csv')
teams[teams['TeamID']==1340]


,TeamID,TeamName,FirstD1Season,LastD1Season
239,1340,Portland St,1999,2026


In [214]:
import pandas as pd

def get_calendar_date(season, day_num, seasons_csv_path='/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MSeasons.csv'):
    """
    Converts a Kaggle Season and DayNum back into a standard calendar date.
    """
    # Load the seasons table
    df_seasons = pd.read_csv(seasons_csv_path)
    
    # Filter for the requested season
    season_row = df_seasons[df_seasons['Season'] == season]
    
    if season_row.empty:
        raise ValueError(f"Season {season} not found in {seasons_csv_path}")
    
    # Extract DayZero and convert to datetime
    # Kaggle format is typically MM/DD/YYYY
    day_zero_str = season_row['DayZero'].values[0]
    day_zero = pd.to_datetime(day_zero_str)
    
    # Calculate the actual date
    actual_date = day_zero + pd.Timedelta(days=int(day_num))
    
    return actual_date

# Example Usage:
target_date = get_calendar_date(2026, 149)
print(target_date.strftime('%Y-%m-%d'))

2026-04-01
